In [ ]:
# youtube - https://www.youtube.com/watch?v=zCEJurLGFRk&t=14s
# % pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib gspread
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
from gspread_dataframe import set_with_dataframe

from data import Data
from cache_pandas import timed_lru_cache
import logging

@timed_lru_cache(seconds=None, maxsize=None)
def dataF():
    logging.info("Fetching data in dataF()")
    df = Data()
    vix_data = df.vix_history()
    policy_rate1, policy_rate2, policy_rate3 = df.policy_rate()
    fx = df.forex_exchange()
    cds = df.cds()
    liquidity = df.liquidity()
    gdp_growth = df.gdp_growth()
    logging.info("Completed fetching data in dataF()")
    return (
        vix_data, #use this
        policy_rate1, #use this
        policy_rate2,
        policy_rate3,
        fx, #use this
        cds, #use this
        liquidity, #use this
        gdp_growth, #use this
    )


vix, policy_rate1, policy_rate2, policy_rate3, fx, cds, liquidity, gdp_growth, = dataF()

scopes = [
    'https://www.googleapis.com/auth/spreadsheets']

creds = Credentials.from_service_account_file(
    'credential.json', scopes=scopes)

client = gspread.authorize(creds)

# 1PP6gpBcoOHjjgCx7LuHLa3dv4ET6ufKvpSY4UDvBczQ
sheet_id = '11Ora6_5EoQJdgnUpjjZgFZyrILguo1c32mde_uQwupw'
sheet = client.open_by_key(sheet_id)

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 40]
})

set_with_dataframe(sheet.sheet1, df) 

values_list = sheet.sheet1.get_all_values()
print(values_list)

worksheet_list = map(lambda x: x.title, sheet.worksheets())
worksheet_name = {'vix_data': vix, 'policy_rate1': policy_rate1, 'fx': fx, 'cds': cds, 'liquidity': liquidity, 'gdp_growth': gdp_growth}
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet1 = sheet.worksheet(new_worksheet_name)
    else:
        sheet1 = sheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)
    set_with_dataframe(sheet1, data)

# sheet.worksheet(new_worksheet_name) -- to select worksheet page
# sheet.add_worksheet(new_worksheet_name, rows=10, cols=10) -- add new worksheet page

c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:161: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)
c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a p

[['Name', 'Age'], ['Alice', '25'], ['Bob', '30'], ['Charlie', '40']]


In [10]:
import pandas as pd

df = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Phillipines\PH IMF BOP Annual.xlsx")
df[df.columns[0]]

0                                   BoP: Current Account
1                           BoP: Current Account: Credit
2                            BoP: Current Account: Debit
3                            BoP: Current Account: Goods
4                    BoP: Current Account: Goods: Credit
                             ...                        
249    BoP: Financial Account: Official Reserve Asset...
250    BoP: Financial Account: Official Reserve Asset...
251    BoP: Financial Account: Official Reserve Asset...
252                           BoP: Exceptional Financing
253                          BoP: Net Errors & Omissions
Name: Select this link and click Refresh/Edit Download to update data and add or remove series, Length: 254, dtype: object

In [235]:
import pandas as pd
import os
from datetime import datetime

df_init = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='BoP edit')

path_dict = dict()
list_1 = ['IMF BOP Annual.xlsx', 'IMF BOP Quarterly.xlsx']
directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[1]))
    # print(v[1]) ## to check the file name
df_init = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='BoP edit')

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['desc'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(2019, 1, 1)]]

init_dict = df_init.to_dict()
df_dict = dict()
for x in init_dict['desc'].values():
    df_temp = df_all_annual_2[(df_all_annual_2[df_all_annual_2.columns[0]] == x)].reset_index(drop=True)
    df_dict[x] = df_temp




In [ ]:
#checking

temp_list = []

for x in df_init.desc.to_list():
    # print(len(df_dict[x]), x) if len(df_dict[x]) == 0 else print(x)
    temp_list.append(len(df_dict[x]))

print((temp_list))
# df_dict['BoP: Financial Account: Other Investment: Liabilities: Other Accounts Receivable/Payable'].head(2)
# df_init.desc.to_list()

[16, 14, 14, 17, 13, 13, 11, 17, 7, 16, 15, 8, 13, 16, 17, 17, 17, 17, 17, 17, 17, 16, 16, 17, 14, 13, 11, 17, 6, 16, 16, 8, 14, 15, 14]


In [390]:
df_all_annual_3 = df_all_annual_2.copy()
df_all_annual_3.drop(columns=['Last Update Time', 'Unit'], inplace=True)
df_all_annual_3.fillna(0, inplace=True)
df_all_annual_3 = df_all_annual_3.melt(id_vars=['Type', 'Region'], var_name='Quarter', value_name='Value')
df_all_annual_3['Quarter'] = df_all_annual_3['Quarter'].astype(str)

df_all_annual_3['Year'] = df_all_annual_3['Quarter'].str.extract(r'(\d{4})')
df_all_annual_3['QuarterNum'] = df_all_annual_3['Quarter'].str.extract(r'-([0-9]{2})-')[0].astype(int)
df_all_annual_3['Half'] = df_all_annual_3['QuarterNum'].apply(lambda x: '1H' if x <= 6 else '2H')

# df_all_annual_3 = df_all_annual_3.groupby(['Region', 'Type', 'Year', 'Half'])['Value'].sum().div(1000).reset_index()
# df_all_annual_3 = df_all_annual_3.pivot(columns=['Region', 'Type','Half', 'Year'], values='Value')
# df_all_annual_3.columns = [f'{half}{year}' for half, year in df_all_annual_3.columns]

# df_all_annual_3[(df_all_annual_3['Type'] == 'BoP: Current Account')&(df_all_annual_3['Region'] == 'Malaysia')]
df_all_annual_3

,Type,Region,Quarter,Value,Year,QuarterNum,Half
0,BoP: Current Account,Brunei,2019-03-01 00:00:00,0.0,2019,3,1H
1,BoP: Current Account: Goods,Brunei,2019-03-01 00:00:00,0.0,2019,3,1H
2,BoP: Current Account: Services,Brunei,2019-03-01 00:00:00,0.0,2019,3,1H
3,BoP: Current Account: Primary Income,Brunei,2019-03-01 00:00:00,0.0,2019,3,1H
4,BoP: Current Account: Secondary Income,Brunei,2019-03-01 00:00:00,0.0,2019,3,1H
...,...,...,...,...,...,...,...
13549,BoP: Financial Account: Other Investment: Liab...,Vietnam,2025-09-01 00:00:00,0.0,2025,9,2H
13550,BoP: Financial Account: Other Investment: Liab...,Vietnam,2025-09-01 00:00:00,0.0,2025,9,2H
13551,BoP: Financial Account: Other Investment: Liab...,Vietnam,2025-09-01 00:00:00,0.0,2025,9,2H
13552,BoP: Financial Account: Other Investment: Liab...,Vietnam,2025-09-01 00:00:00,0.0,2025,9,2H


In [443]:
# df_all_annual_2[df_all_annual_2.columns[4:]]
# (df_all_annual_2[x] + df_all_annual_2[x+1]).div(1000)
num = 0
df_col = df_all_annual_2[df_all_annual_2.columns[4:]].columns
df_temp_col_1 = df_all_annual_2[df_all_annual_2.columns[:4]].reset_index(drop=True)

for x in range(len(df_col)//2):
    df_col_temp = (df_all_annual_2[df_col[num]] + df_all_annual_2[df_col[num + 1]]).div(1000)
    # df_temp_col_1['Type'] = df_col_temp
    num += 2

df_temp_col_1

,Type,Region,Unit,Last Update Time
0,BoP: Current Account,Brunei,USD mn,2024-10-24
1,BoP: Current Account: Goods,Brunei,USD mn,2024-10-24
2,BoP: Current Account: Services,Brunei,USD mn,2024-10-24
3,BoP: Current Account: Primary Income,Brunei,USD mn,2024-10-24
4,BoP: Current Account: Secondary Income,Brunei,USD mn,2024-10-24
...,...,...,...,...
497,BoP: Financial Account: Other Investment: Liab...,Vietnam,USD mn,2024-11-21
498,BoP: Financial Account: Other Investment: Liab...,Vietnam,USD mn,2024-11-21
499,BoP: Financial Account: Other Investment: Liab...,Vietnam,USD mn,2024-11-21
500,BoP: Financial Account: Other Investment: Liab...,Vietnam,USD mn,2024-11-21


In [ ]:
(df_all_annual_2[df_col[num]] + df_all_annual_2[df_col[num + 1]]).div(1000)

0    NaN
3    NaN
10   NaN
73   NaN
82   NaN
      ..
34   NaN
35   NaN
39   NaN
45   NaN
49   NaN
Length: 502, dtype: float64

In [375]:
import pandas as pd

# Example input with multiple years (just 2019 shown here)
df = pd.DataFrame({
    '1Q2019': [921.52],
    '2Q2019': [932.19],
    '3Q2019': [897.25],
    '4Q2019': [912.07],
    '1Q2020': [950.00],
    '2Q2020': [960.00],
    '3Q2020': [940.00],
    '4Q2020': [930.00],
}, index=['Cambodia'])

# Step 1: Melt to long format
df_long = df.reset_index().melt(id_vars='index', var_name='Quarter', value_name='Value')

# Step 2: Extract Year and Half
df_long['Year'] = df_long['Quarter'].str.extract(r'(\d{4})')
df_long['QuarterNum'] = df_long['Quarter'].str.extract(r'(\d)Q').astype(int)
df_long['Half'] = df_long['QuarterNum'].apply(lambda x: '1H' if x <= 2 else '2H')

# Step 3: Group by country + year + half
df_half = df_long.groupby(['index', 'Year', 'Half'])['Value'].sum().div(1000).reset_index()

# Step 4: Pivot back to wide format
df_final = df_half.pivot(index='index', columns=['Half', 'Year'], values='Value')

# Optional: flatten MultiIndex columns
# df_final.columns = [f'{half}{year}' for half, year in df_final.columns]
# df_final = df_final.reset_index()

df_final

Half,1H,2H,1H,2H
Year,2019,2019,2020,2020
index,,,,
Cambodia,1.85371,1.80932,1.91,1.87


In [345]:
df_long

,index,Quarter,Value,Year,QuarterNum,Half
0,Cambodia,1Q2019,921.52,2019,1,1H
1,Cambodia,2Q2019,932.19,2019,2,1H
2,Cambodia,3Q2019,897.25,2019,3,2H
3,Cambodia,4Q2019,912.07,2019,4,2H
4,Cambodia,1Q2020,950.00,2020,1,1H
5,Cambodia,2Q2020,960.00,2020,2,1H
6,Cambodia,3Q2020,940.00,2020,3,2H
7,Cambodia,4Q2020,930.00,2020,4,2H


In [259]:
df_long

,index,Quarter,Value,Year,QuarterNum,Half
0,Cambodia,1Q2019,921.52,2019,1,1H
1,Cambodia,2Q2019,932.19,2019,2,1H
2,Cambodia,3Q2019,897.25,2019,3,2H
3,Cambodia,4Q2019,912.07,2019,4,2H
4,Cambodia,1Q2020,950.00,2020,1,1H
5,Cambodia,2Q2020,960.00,2020,2,1H
6,Cambodia,3Q2020,940.00,2020,3,2H
7,Cambodia,4Q2020,930.00,2020,4,2H
